**Sentiments analysis using Neural Network**

In [1]:
#%pip install --upgrade nltk

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
import re
import nltk
# using the variable sw to hold all stopwords that are in English
nltk.download('stopwords')
sw = stopwords.words('english')

[nltk_data] Downloading package stopwords to C:\Users\oyelola
[nltk_data]     Ibrahim\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [6]:
ds = pd.read_csv("C:/Users/oyelola Ibrahim/Desktop/googleplaystore_user_reviews.csv")

In [7]:
ds.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [8]:
ds.info()

<class 'pandas.DataFrame'>
RangeIndex: 64295 entries, 0 to 64294
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   App                     64295 non-null  str    
 1   Translated_Review       37427 non-null  str    
 2   Sentiment               37432 non-null  str    
 3   Sentiment_Polarity      37432 non-null  float64
 4   Sentiment_Subjectivity  37432 non-null  float64
dtypes: float64(2), str(3)
memory usage: 2.5 MB


In [10]:
# number of elememnts before removing the NAN values
print("Size before removing Nan: %s" % len(ds))

# number of elememnts after removing the NAN values
ds.dropna(axis=0, inplace=True)
print("Size after removing Nan: %s" % len(ds))

Size before removing Nan: 64295
Size after removing Nan: 37427


In [14]:
def cleaning_data(data):
    aux_list = []
    flag = False
    for phase_word in data:
        word_list = []
        for word in phase_word.split():
            word = word.lower()
            if flag and not word in sw:
                flag = False
                word_list.append('not_'+word)
                continue
            if re.search('(n\'t)$|(not)|(no)|(never)', word):
                flag = True
                continue
            if not word in sw:
                word = re.sub('[w_0-9]', ' ', word)
                word_list.append(word)
        aux_list.append(' '.join(word_list))
    return aux_list

In [15]:
ds.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000
5,10 Best Foods for You,Best way,Positive,1.00,0.300000


In [16]:
X = cleaning_data(ds["Translated_Review"])
y = ds["Sentiment"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

In [17]:
# This countVectorizer is used to represent the words as a list of values, instead of text
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()

vectorizer.fit(X)
X_train = vectorizer.transform(X_train)
X_test = vectorizer.transform(X_test)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(y)
y_train = le.transform(y_train)
y_test = le.transform(y_test)

In [22]:
from keras.models import Sequential
from keras.layers import Dense

model = Sequential()
model.add(Dense(units=100, activation='relu', input_dim=len (vectorizer.get_feature_names_out())))
model.add(Dense(units=3, activation='sigmoid')) 
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=2, verbose=1,)
scores = model.evaluate(X_test, y_test, verbose = 1)
print("Accuracy:", scores[1])

Epoch 1/2
936/936 ━━━━━━━━━━━━━━━━━━━━ 22s 22ms/step - accuracy: 0.8455 - loss: 0.4126
Epoch 2/2
936/936 ━━━━━━━━━━━━━━━━━━━━ 21s 22ms/step - accuracy: 0.9576 - loss: 0.1368
234/234 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9150 - loss: 0.2585
Accuracy: 0.9150413870811462
